# shap-relativities

**Extract multiplicative rating relativities from GBM models — in the factor-table format rating engines expect.**

Your CatBoost model outperforms your GLM. But the regulator wants a factor table, and `exp(SHAP)` gives you one. This notebook shows the full workflow: fit a Poisson CatBoost model on synthetic motor data, extract relativities, compare them to the known true parameters.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/burning-cost/shap-relativities/blob/main/notebooks/quickstart.ipynb)

In [ ]:
!pip install -q "shap-relativities[ml]" catboost polars

## Synthetic motor data

We use the built-in `load_motor()` dataset — 20,000 synthetic UK motor policies with a known Poisson DGP.
The true parameters are exported via `TRUE_FREQ_PARAMS`, so we can measure how accurately SHAP recovers them.

In [ ]:
import polars as pl
import catboost
from shap_relativities.datasets.motor import load_motor, TRUE_FREQ_PARAMS

df = load_motor(n_policies=20_000, seed=42)
df = df.with_columns([
    ((pl.col("conviction_points") > 0).cast(pl.Int32)).alias("has_convictions"),
    pl.col("area").replace({"A": "0", "B": "1", "C": "2", "D": "3", "E": "4", "F": "5"})
      .cast(pl.Int32).alias("area_code"),
])

print(f"Portfolio: {df.shape[0]:,} policies")
print(f"Claim frequency: {(df['claim_count'].sum() / df['exposure'].sum()):.4f} claims/year")
print(f"True NCD coefficient: {TRUE_FREQ_PARAMS}")
df.head(4)

## Fit a Poisson CatBoost model

Three rating factors: area band (6 levels), NCD years (0–5), conviction indicator.
CatBoost Poisson with exposure as weight.

In [ ]:
features = ["area_code", "ncd_years", "has_convictions"]
X = df.select(features)

pool = catboost.Pool(
    data=X.to_pandas(),
    label=df["claim_count"].to_numpy(),
    weight=df["exposure"].to_numpy(),
)
model = catboost.CatBoostRegressor(
    loss_function="Poisson",
    iterations=300,
    learning_rate=0.05,
    depth=6,
    random_seed=42,
    verbose=0,
)
model.fit(pool)
print("Model fitted.")

## Extract relativities with SHAP

`SHAPRelativities` computes TreeSHAP values and converts them to multiplicative relativities.
The maths: for a Poisson GBM with log link, SHAP values are additive in log space.
The relativity for level k vs base level 0 is `exp(mean_shap(k) - mean_shap(0))` — directly analogous to `exp(beta_k - beta_0)` from a GLM.

In [ ]:
from shap_relativities import SHAPRelativities

sr = SHAPRelativities(
    model=model,
    X=X,
    exposure=df["exposure"],
    categorical_features=features,
)
sr.fit()

rels = sr.extract_relativities(
    normalise_to="base_level",
    base_levels={"area_code": 0, "ncd_years": 0, "has_convictions": 0},
)
print(rels.select(["feature", "level", "relativity", "lower_ci", "upper_ci"]))

## Validation and accuracy check

Two checks:
1. **Reconstruction**: `exp(shap_values.sum(axis=1) + expected_value)` must match model predictions to within 1e-4. If this fails, the explainer was constructed incorrectly.
2. **Relativity accuracy**: Compare to the known true parameters.

In [ ]:
checks = sr.validate()
print("Reconstruction check:", checks["reconstruction"])
print("Sparse levels check:", checks["sparse_levels"])

import numpy as np
# NCD=5 relativity vs true exp(-0.6) = 0.549
ncd5 = rels.filter((pl.col("feature") == "ncd_years") & (pl.col("level") == "5"))
ncd5_rel = float(ncd5["relativity"][0])
true_ncd5 = float(np.exp(-0.12 * 5))  # from TRUE_FREQ_PARAMS
print(f"\nNCD=5 relativity: SHAP={ncd5_rel:.3f}, True={true_ncd5:.3f}, Error={abs(ncd5_rel - true_ncd5)/true_ncd5:.1%}")

# Conviction loading
conv = rels.filter((pl.col("feature") == "has_convictions") & (pl.col("level") == "1"))
conv_rel = float(conv["relativity"][0])
print(f"Conviction loading: SHAP={conv_rel:.3f}")

## What you should see

- Reconstruction check: PASS (max error < 1e-4). This verifies SHAP values correctly decompose the model's predictions.
- NCD=5 relativity: approximately 0.43–0.55. The true value is 0.549 (`exp(-0.6)`). SHAP errors of 10–25% are typical — the GBM does not constrain to log-linear form, so relativities are less precise than GLM `exp(beta)` but the model is more accurate overall.
- Conviction loading: approximately 1.5–1.7×.

The output format — one row per (feature, level) with confidence intervals — maps directly to a rating engine's factor table CSV import.

## Next steps

- **`extract_continuous_curve()`** — smoothed relativity curve for continuous features (driver age, annual mileage) with optional isotonic monotonicity enforcement
- **`feature_perturbation="interventional"`** — correct for correlated features (e.g. postcode and socioeconomic index)
- **`rels.write_csv("relativities.csv")`** — export directly for Radar, Emblem, or Earnix factor table import

**GitHub:** https://github.com/burning-cost/shap-relativities  
**PyPI:** https://pypi.org/project/shap-relativities/